## Session 3 · Notebook 3 — Data Summarisation (Python)

**Dataset:** `Y1_T1_2025.xlsx` — Term 1, 2025 first-year unit-attempt extract.

So far you've summarised *one whole column*. Real questions are usually **per
group**: average mark *by program*, attempts *by unit*, pass patterns *by gender*.
This notebook covers the pandas **split–apply–combine** toolkit:

- **`groupby()`** — split rows into groups by a column
- **`agg()`** — apply one or several summary statistics to each group
- **`sort_values()`** — order the result to rank groups

This solution copy shows one possible completed version.

### 1. Load the data

Read the **`Extract`** sheet into `df` and print its shape.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_FOLDER = Path("../data")
DATA_FILE = DATA_FOLDER / "Y1_T1_2025.xlsx"

df = pd.read_excel(DATA_FILE, sheet_name="Extract")
print(df.shape)

### 2. Group by one column, one statistic

Compute the **mean `mark` for each `program_code`**. The pattern is
`df.groupby("KEY")["VALUE"].STAT()`. Round to 2 decimals.

In [ ]:
df.groupby("program_code")["mark"].mean().round(2)

### 3. Counting rows per group

Often you want *how many*, not an average. `.size()` counts rows per group. Count
the **attempts per `unit_code`**, and use `sort_values()` to put the busiest unit
first.

In [ ]:
df.groupby("unit_code").size().sort_values(ascending=False)

### 4. Several statistics at once with `agg()`

`agg()` applies a **list** of statistics in one call. Group by `gender` and
compute the `count`, `mean`, `median` and `std` of `mark` together (round to 2).

In [ ]:
df.groupby("gender")["mark"].agg(["count", "mean", "median", "std"]).round(2)

### 5. Named aggregations (tidy column names)

`agg(newname=("column", "stat"))` gives each result a clear name. Build a
per-`program_code` summary with three columns: `n` (group size), `mean_mark` and
`median_mark`.

In [ ]:
df.groupby("program_code").agg(
    n=("mark", "size"),
    mean_mark=("mark", "mean"),
    median_mark=("mark", "median"),
).round(2)

### 6. Ranking with `sort_values()`

Grouped results come back sorted by the *group key*. To **rank**, sort by the
*value*. Take the mean mark per `program_code` and sort it high-to-low. Which
program has the highest average mark? The lowest?

In [ ]:
df.groupby("program_code")["mark"].mean().round(2).sort_values(ascending=False)

### 7. Put it together — a mini report

For each `unit_code`, build a named-agg summary with `attempts` (size),
`mean_mark`, and `pass_rate` — the share of attempts with `mark >= 50` — then sort
by `mean_mark` descending.

> Hint: make a boolean column `df["passed"] = df["mark"] >= 50`, then take its
> `mean` per group (the mean of a True/False column is the proportion of `True`).

In [ ]:
df["passed"] = df["mark"] >= 50
df_report = df.groupby("unit_code").agg(
    attempts=("mark", "size"),
    mean_mark=("mark", "mean"),
    pass_rate=("passed", "mean"),
).round(2).sort_values("mean_mark", ascending=False)
df_report

### 8. Wrap-up

In your own words (2–3 sentences): which grouping told you the most, and why is a
group *mean* on its own sometimes misleading (hint: think about what else differs
between the groups)?